# CosyVoice 단독 테스트

CosyVoice3 설치(Drive 캐시) -> Drive 음성 로드 -> 클로닝 -> 결과 저장

**런타임: GPU (T4 이상)**


## Step 1. GPU 확인


In [ ]:
import torch
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: GPU 없음")


## Step 2. Google Drive 마운트

음성 파일(.wav)을 Drive에 미리 올려두세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ★ 본인 경로로 수정
AUDIO_PATH = "/content/drive/MyDrive/test_audio.wav"
OUTPUT_DIR = "/content/drive/MyDrive/cosyvoice_test_output"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("입력 파일 존재:", os.path.exists(AUDIO_PATH))
print("출력 폴더:", OUTPUT_DIR)


## Step 3. CosyVoice 설치 (Drive 캐시)

처음 실행 시 Drive에 저장합니다. 이후 실행은 Drive에서 복원해서 빠릅니다.


In [ ]:
import os, shutil
COSYVOICE_ROOT  = "/content/CosyVoice"
COSYVOICE_DRIVE = "/content/drive/MyDrive/CosyVoice"

if os.path.exists(COSYVOICE_DRIVE):
    print("Drive 캐시에서 복원 중...")
    if os.path.exists(COSYVOICE_ROOT):
        shutil.rmtree(COSYVOICE_ROOT)
    shutil.copytree(COSYVOICE_DRIVE, COSYVOICE_ROOT)
    print("[OK] 복원 완료")
else:
    print("처음 설치 중 (git clone + 모델 다운로드)...")
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSYVOICE_ROOT}
    from huggingface_hub import snapshot_download
    snapshot_download("FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=f"{COSYVOICE_ROOT}/pretrained_models/Fun-CosyVoice3-0.5B-2512")
    print("Drive에 백업 중...")
    if os.path.exists(COSYVOICE_DRIVE):
        shutil.rmtree(COSYVOICE_DRIVE)
    shutil.copytree(COSYVOICE_ROOT, COSYVOICE_DRIVE)
    print("[OK] Drive 백업 완료")

!pip install -q --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ -r {COSYVOICE_ROOT}/requirements.txt
print("[OK] 패키지 설치 완료")


## Step 4. CosyVoice 로드


In [ ]:
import sys, time
sys.path.insert(0, COSYVOICE_ROOT)
sys.path.insert(0, f"{COSYVOICE_ROOT}/third_party/Matcha-TTS")

from cosyvoice.cli.cosyvoice import CosyVoice3

t0 = time.time()
model = CosyVoice3(f"{COSYVOICE_ROOT}/pretrained_models/Fun-CosyVoice3-0.5B-2512")

if hasattr(model, "model") and hasattr(model.model, "llm"):
    model.model.llm = model.model.llm.float()
    print("LLM float32 변환 완료")

print(f"[OK] CosyVoice3 로드 완료 ({time.time()-t0:.1f}s)")
print(f"sample_rate: {model.sample_rate}")


## Step 5. 음성 로드 + 클로닝


In [ ]:
import numpy as np, soundfile as sf, tempfile, os, time
from math import gcd
from scipy.signal import resample_poly

SR = 16000
data, src_sr = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
audio = data.mean(axis=1).astype("float32")
if src_sr != SR:
    g = gcd(SR, src_sr)
    audio = resample_poly(audio, SR // g, src_sr // g).astype("float32")
audio = np.clip(audio, -1.0, 1.0)
print(f"입력 음성: {len(audio)/SR:.2f}초")

PROMPT_TEXT = input("레퍼런스 음성에서 실제로 하는 말 : ")
TTS_TEXT    = input("새로 합성할 문장 : ")

with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
    sf.write(tmp.name, audio, SR)
    ref_path = tmp.name

end_of_prompt = "<|endofprompt|>"
prompt_text = PROMPT_TEXT if PROMPT_TEXT.endswith(end_of_prompt) else PROMPT_TEXT + end_of_prompt

print("클로닝 중...")
t0 = time.time()
chunks = []
for result in model.inference_zero_shot(TTS_TEXT, prompt_text, ref_path, stream=False):
    chunks.append(result["tts_speech"].squeeze().numpy())
os.unlink(ref_path)

cloned = np.concatenate(chunks).astype("float32")
cloned = np.clip(cloned, -1.0, 1.0)
print(f"클로닝 완료 ({time.time()-t0:.1f}s) -> {len(cloned)/model.sample_rate:.2f}초")


## Step 6. 결과 저장 + 청취


In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from pathlib import Path

stem = Path(AUDIO_PATH).stem
out_original = f"{OUTPUT_DIR}/{stem}_original.wav"
out_cloned   = f"{OUTPUT_DIR}/{stem}_cloned.wav"

sf.write(out_original, audio, SR)
sf.write(out_cloned,   cloned, model.sample_rate)
print("저장 완료:")
print(f"  원본 : {out_original}")
print(f"  클론 : {out_cloned}")

print("=== 원본 ===")
display(Audio(audio, rate=SR))
print("=== 클론 (CosyVoice3) ===")
display(Audio(cloned, rate=model.sample_rate))
